# Modeling

After completing data exploration, feature analysis, and preprocessing in the previous notebooks, this notebook focuses on building the first set of machine‑learning models for the sentiment classification task.

The prepared dataset is loaded, split into training and test sets, and used to train multiple classical baseline classifiers. Their performance is evaluated using standard metrics, and both the trained models and the corresponding evaluation results are stored for use in subsequent analysis.

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve()
while not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.append(str(project_root))

In [2]:
from src.utils.loaders import load_config
import pandas as pd
from sklearn.model_selection import train_test_split


from src.models.train_model import train_model
from src.models.evaluate_model import evaluate_model, compute_learning_curve
from src.models.feature_importance import get_feature_importance
from src.utils.model_repository import ModelRepository

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

## Importing the Cleaned Dataset and Creating Train/Test Splits

The preprocessed dataset is imported based on the paths and column definitions stored in the project configuration. After loading the text and label fields, the data is split into training and test sets according to the configured split ratio and random seed, providing a reproducible foundation for model training.

In [3]:
config = load_config("configs/config.yaml")

data_path = config["data"]["preprocessed_path"]
text_col = config["data"]["text_column"]
label_col = config["data"]["label_column"]

df = pd.read_csv(data_path, keep_default_na=False)
df.head()

X = df[text_col]
y = df[label_col]

In [4]:
test_size = config["training"]["test_size"]
random_state = config["training"]["random_state"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)

## Baseline Model Training and Experiment Logging

This section trains several classical baseline models using a TF‑IDF representation of the text data. TF‑IDF (Term Frequency–Inverse Document Frequency) converts the preprocessed documents into weighted numerical feature vectors by emphasizing terms that are informative for distinguishing between documents while reducing the influence of very frequent, less meaningful words. This representation is widely used in traditional machine‑learning pipelines for text classification because it is efficient, interpretable, and well suited for high‑dimensional sparse data.

Based on these TF‑IDF features, three established classifiers are trained. Logistic Regression serves as a linear model that estimates class boundaries through probabilistic decision functions and often performs strongly on text classification tasks due to its robustness in high‑dimensional spaces. The Linear Support Vector Machine classifier maximizes the margin between classes and is particularly effective for sparse feature representations such as TF‑IDF. The third model, Multinomial Naive Bayes, follows a probabilistic approach that assumes conditional independence between features. Despite this simplifying assumption, it is computationally efficient and frequently provides competitive performance on bag‑of‑words and TF‑IDF representations.

All models are trained using the same TF‑IDF configuration to ensure comparability. After training, each classifier is evaluated on the test set, and feature‑importance visualizations are generated where applicable. The trained pipelines, evaluation metrics, configuration details, and visual outputs are then stored as experiment artifacts for subsequent comparison and analysis.

In [5]:
tfidf_config = config["tfidf"]

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        C=1.0,
        class_weight="balanced",
        random_state=42
    ),
    "Linear SVM": LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=42
    ),
    "Naive Bayes": MultinomialNB()
}

In [9]:
repo = ModelRepository()

for name, model in models.items():
    print(f"Training {name}...")

    pipeline = train_model(model, X_train, y_train, max_features=tfidf_config["max_features"], ngram_range=tuple(tfidf_config["ngram_range"]))

    metrics, cm = evaluate_model(pipeline, X_test, y_test)

    feature_importance_df = get_feature_importance(pipeline, 20)

    learning_curve_df = compute_learning_curve(pipeline, X, y)

    model_config = {
        "model": name,
        "params": model.get_params(),
        "tfidf": tfidf_config
    }

    repo.save_experiment(
        model_name=name,
        pipeline=pipeline,
        metrics=metrics,
        config=model_config,
        feature_importance_df = feature_importance_df,
        confusion_matrix=cm,
        learning_curve_df=learning_curve_df
    )

Training Logistic Regression...
              precision    recall  f1-score   support

    negative       0.70      0.80      0.75        61
     neutral       0.93      0.94      0.93       278
    positive       0.81      0.74      0.77       114

    accuracy                           0.87       453
   macro avg       0.81      0.83      0.82       453
weighted avg       0.87      0.87      0.87       453

Experiment saved to: /home/melanie/ml_projects/financial-phrase-sentiment-analysis/models/logistic_regression
Training Linear SVM...
              precision    recall  f1-score   support

    negative       0.74      0.84      0.78        61
     neutral       0.95      0.96      0.96       278
    positive       0.85      0.77      0.81       114

    accuracy                           0.90       453
   macro avg       0.85      0.86      0.85       453
weighted avg       0.90      0.90      0.90       453

Experiment saved to: /home/melanie/ml_projects/financial-phrase-sentiment